# **ResNet50 + Self-Attention (SA) Module — EXP03**

Notebook ini adalah **turunan langsung dari `exp01-cnn-baseline.ipynb` (ResNet50 Baseline Final)**.
Semua komponen training regime (scheduler `ReduceLROnPlateau`, early stopping berbasis `val_f1`,
augmentasi "medium", AMP, seed control, class weight ternormalisasi, K-Fold, format logging W&B)
**dipertahankan 100% identik** dengan baseline -- supaya perbandingan EXP01 (baseline) vs EXP03 (SA)
adil, tanpa confound training regime.

**Satu-satunya perubahan** ada di arsitektur model:
- Baseline (EXP01): `timm.create_model("resnet50", ...)` langsung -> global pool -> FC head bawaan timm.
- EXP03 (notebook ini): backbone ResNet50 yang sama (feature extractor, `forward_features`) diikuti
  **modul Self-Attention 2D (non-local / SAGAN-style)** yang beroperasi di atas feature map
  `[B, 2048, 7, 7]` sebelum global average pooling -> FC head.

**Ide dasar SA module**: tiap posisi spasial pada feature map dibiarkan "melihat" semua posisi
spasial lain (bukan cuma tetangga lokal seperti konvolusi biasa), lalu menghitung bobot relevansi
(attention) antar posisi tersebut. Ini membantu model menangkap hubungan/kontras jangka panjang
antar area kulit (misal: pola lesi yang tersebar, tekstur di tepi vs tengah lesi) yang sulit
ditangkap murni oleh receptive field konvolusi lokal ResNet.


## 1. Import & Setup

In [1]:
# 1. Install & Import
import os, copy, random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
# AMP -- sebagian besar operasi forward jalan di float16 (lebih cepat & hemat VRAM
# di GPU RTX/Tensor Core), backward/update tetap presisi lewat GradScaler.
from torch.cuda.amp import autocast, GradScaler

from torchvision import transforms, datasets
from PIL import Image
from tqdm import tqdm

import timm   # 1 API dipakai semua arsitektur (ResNet18/50, EfficientNet, ViT, dst)
import wandb

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

# Ambil API key dari environment variable
wandb_api_key = os.getenv("WANDB_API_KEY")

os.environ["TORCH_HOME"] = "D:/cache/torch"
os.environ["HF_HOME"] = "D:/cache/huggingface"

os.environ["WANDB_DIR"] = "D:/cache/wandb"
os.environ["WANDB_CACHE_DIR"] = "D:/cache/wandb_cache"

os.environ["TEMP"] = "D:/cache/temp"
os.environ["TMP"] = "D:/cache/temp"

os.environ["CUDA_CACHE_PATH"] = "D:/cache/cuda"

print(os.getcwd())

# Login ke wandb
wandb.login(key=wandb_api_key)

# Seed eksplisit -- konsisten dengan baseline: nge-seed StratifiedKFold, init bobot
# FC head / attention module, dan urutan shuffle.
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# cudnn.benchmark auto-tune algoritma konvolusi tercepat untuk ukuran input yang
# konsisten (semua di-resize ke 224x224).
torch.backends.cudnn.benchmark = True


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\UNIDA\_netrc.


d:\Devianest_SkripsiTest\Code_CNN


wandb: Currently logged in as: devianestnarendra to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Device: cuda


## 2. Config

In [2]:
TRAIN_DIR = r"D:\Devianest_SkripsiTest\train"
TEST_DIR  = r"D:\Devianest_SkripsiTest\test"

OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# PERBAIKAN (EXP03 vs baseline): nama arsitektur diganti supaya jelas beda dari
# baseline EXP01 -- backbone timm tetap sama persis ("resnet50"), yang berubah
# hanya HEAD di atasnya (lihat bagian "Self-Attention Module" di bawah).
ARCH_KEY  = "EXP03_ResNet50_SelfAttention"
TIMM_NAME = "resnet50"

IMG_SIZE     = 224
BATCH_SIZE   = 32          # tetap sama seperti baseline (ResNet50 lebih berat, batch lebih kecil)
EPOCHS       = 50
N_FOLDS      = 5
DROPOUT      = 0.3         # dropout pada head, dipertahankan sama seperti baseline
LR           = 1e-4        # dipertahankan sama seperti baseline (bukan confound tambahan)
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
EARLY_STOP_PATIENCE = 7

# PERBAIKAN (EXP03 baru): reduction ratio untuk query/key conv di SA module --
# proj_dim = feat_dim // SA_REDUCTION. Nilai 8 adalah nilai umum dari paper
# Self-Attention GAN (Zhang et al., 2019) -- cukup untuk menekan cost O(N^2) attention
# tanpa terlalu membatasi kapasitas representasi query/key.
SA_REDUCTION = 8

WANDB_PROJECT = "SkinDisease-CNN"   # tetap 1 project W&B yang sama dengan baseline

# Scope unfreeze backbone tetap sama dengan baseline (layer2/3/4) supaya kapasitas
# backbone yang dibandingkan konsisten -- "attention" ditambahkan karena modul baru
# ini WAJIB selalu trainable (bobotnya random-init, belum di-pretrain).
UNFREEZE_PATTERNS = ["layer2", "layer3", "layer4", "attention", "fc"]

# Scheduler ReduceLROnPlateau -- identik dengan baseline, tidak ada confound tambahan
# di training regime, supaya selisih hasil murni dari modul attention.
SCHEDULER_FACTOR    = 0.1
SCHEDULER_PATIENCE  = 2
SCHEDULER_THRESHOLD = 1e-4
SCHEDULER_MIN_LR    = 1e-7


## 3. Dataset & Augmentasi

In [3]:
# Augmentasi disamakan PERSIS dengan baseline -- bukan bagian yang diuji di EXP03,
# supaya perbandingan EXP01 vs EXP03 murni soal arsitektur (SA module), bukan
# preprocessing.
def get_transforms(img_size):
    train_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
        transforms.RandomResizedCrop(img_size, scale=(0.8, 1.0)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    eval_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    return train_tf, eval_tf


classes = sorted(os.listdir(TRAIN_DIR))
class_to_idx = {c: i for i, c in enumerate(classes)}
num_classes = len(classes)

filepaths, labels = [], []
for label in classes:
    class_path = os.path.join(TRAIN_DIR, label)
    for img in os.listdir(class_path):
        filepaths.append(os.path.join(class_path, img))
        labels.append(class_to_idx[label])

print("Total Images :", len(filepaths))
print("Classes      :", num_classes)


class SkinDataset(Dataset):
    def __init__(self, filepaths, labels, transform=None):
        self.filepaths = filepaths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        image = Image.open(self.filepaths[idx]).convert("RGB")
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, label


Total Images : 15557
Classes      : 23


## 4. Early Stopping (val_f1) + K-Fold

In [4]:
class EarlyStopping:
    # Kriteria val_f1 (bukan val_loss) -- konsisten dengan baseline: lebih robust
    # untuk data imbalanced (23 kelas DermNet) dibanding val_loss.
    def __init__(self, patience=5):
        self.patience = patience
        self.best_f1 = -np.inf
        self.counter = 0

    def step(self, val_f1):
        if val_f1 > self.best_f1:
            self.best_f1 = val_f1
            self.counter = 0
            return False
        self.counter += 1
        return self.counter >= self.patience


skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)


## 5. Self-Attention (SA) Module — **BARU di EXP03**

Modul non-local self-attention 2D bergaya SAGAN (Zhang et al., *Self-Attention Generative
Adversarial Networks*, 2019), diadaptasi untuk classification head (bukan generator).

Diberi feature map `x` berukuran `[B, C, H, W]` (di ResNet50, `C=2048`, `H=W=7` untuk input 224x224):

1. **Query** `f(x)` dan **Key** `g(x)`: proyeksi 1x1 conv ke dimensi lebih kecil (`C/r`), lalu
   di-reshape jadi `[B, N, C/r]` dengan `N = H*W` (tiap piksel jadi satu "token").
2. **Attention map**: `softmax(Query · Key^T)` -> matriks `[B, N, N]`, berisi skor relevansi
   antar SEMUA pasangan posisi spasial (bukan cuma tetangga lokal seperti kernel konvolusi).
3. **Value** `h(x)`: proyeksi 1x1 conv ke dimensi penuh `C`, lalu diagregasi memakai attention
   map -> tiap posisi keluaran adalah kombinasi berbobot dari SEMUA posisi input.
4. **Residual gate**: `out = gamma * attention_output + x`, dengan `gamma` parameter skalar
   yang **diinisialisasi 0**. Ini trik penting dari paper SAGAN: di awal training modul attention
   praktis "transparan" (`out ≈ x`, backbone pretrained tidak langsung terganggu bobot attention
   yang masih random), lalu `gamma` dipelajari naik perlahan seiring training kalau attention
   memang membantu.


In [5]:
class SelfAttention2d(nn.Module):
    """Non-local self-attention 2D (gaya SAGAN) untuk feature map CNN.

    Input : x [B, C, H, W]
    Output: out [B, C, H, W] (residual: gamma * attention_output + x),
            attn [B, N, N] (opsional, untuk visualisasi)
    """
    def __init__(self, in_dim, reduction=8):
        super().__init__()
        reduced_dim = max(in_dim // reduction, 1)
        self.query_conv = nn.Conv2d(in_dim, reduced_dim, kernel_size=1)
        self.key_conv   = nn.Conv2d(in_dim, reduced_dim, kernel_size=1)
        self.value_conv = nn.Conv2d(in_dim, in_dim, kernel_size=1)
        # gamma diinisialisasi 0 -- di awal training modul attention "transparan"
        # (out = x), supaya tidak merusak fitur pretrained backbone sebelum
        # attention sempat belajar sesuatu yang berguna.
        self.gamma = nn.Parameter(torch.zeros(1))
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x, return_attention=False):
        B, C, H, W = x.shape
        N = H * W

        proj_query = self.query_conv(x).view(B, -1, N).permute(0, 2, 1)  # [B, N, C/r]
        proj_key   = self.key_conv(x).view(B, -1, N)                     # [B, C/r, N]
        energy     = torch.bmm(proj_query, proj_key)                     # [B, N, N]
        attn       = self.softmax(energy)                                # softmax antar posisi key

        proj_value = self.value_conv(x).view(B, -1, N)                   # [B, C, N]
        out = torch.bmm(proj_value, attn.permute(0, 2, 1))               # [B, C, N]
        out = out.view(B, C, H, W)

        out = self.gamma * out + x

        if return_attention:
            return out, attn
        return out


## 6. Model Builder + Freeze Strategy

In [6]:
class ResNetSA(nn.Module):
    """ResNet50 (timm backbone, sama persis dengan baseline) + SelfAttention2d
    yang disisipkan di atas feature map terakhir (sebelum global average pooling),
    lalu FC head sederhana -- sama filosofinya dengan baseline (drop_rate, bukan
    custom multi-layer head), supaya kapasitas head TIDAK jadi confound tambahan.
    """
    def __init__(self, timm_name, num_classes, dropout=DROPOUT, sa_reduction=SA_REDUCTION):
        super().__init__()
        # num_classes=0, global_pool="" -> backbone mengembalikan feature map
        # [B, C, H, W] mentah lewat forward_features(), TANPA pooling/FC bawaan timm.
        self.backbone = timm.create_model(
            timm_name, pretrained=True, num_classes=0, global_pool=""
        )
        feat_dim = self.backbone.num_features  # 2048 untuk resnet50

        self.attention = SelfAttention2d(feat_dim, reduction=sa_reduction)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(feat_dim, num_classes)

    def forward(self, x, return_attention=False):
        feats = self.backbone.forward_features(x)          # [B, 2048, 7, 7]
        attended = self.attention(feats, return_attention=return_attention)
        if return_attention:
            attended, attn_map = attended

        pooled = self.pool(attended).flatten(1)             # [B, 2048]
        pooled = self.dropout(pooled)
        logits = self.fc(pooled)                             # [B, num_classes]

        if return_attention:
            return logits, attn_map
        return logits


def build_model(num_classes, dropout=DROPOUT, sa_reduction=SA_REDUCTION):
    model = ResNetSA(TIMM_NAME, num_classes, dropout=dropout, sa_reduction=sa_reduction)
    return model


def apply_freeze_strategy(model, patterns=UNFREEZE_PATTERNS):
    for p in model.parameters():
        p.requires_grad = False
    for name, p in model.named_parameters():
        if any(pat in name for pat in patterns):
            p.requires_grad = True

    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model.parameters())
    print(f"  [{ARCH_KEY}] Trainable params: {n_trainable:,} / {n_total:,} ({100*n_trainable/n_total:.1f}%)")
    return model


## 7. Train 1 Fold

In [7]:
def train_one_fold(fold, train_idx, val_idx, run):
    train_tf, eval_tf = get_transforms(IMG_SIZE)

    train_files  = [filepaths[i] for i in train_idx]
    train_labels = [labels[i] for i in train_idx]
    val_files    = [filepaths[i] for i in val_idx]
    val_labels   = [labels[i] for i in val_idx]

    train_loader = DataLoader(
        SkinDataset(train_files, train_labels, train_tf),
        batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True
    )
    val_loader = DataLoader(
        SkinDataset(val_files, val_labels, eval_tf),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True
    )

    # Reseed per fold -- inisialisasi FC head, gamma SA module, & urutan shuffle
    # reproducible, tidak tergantung urutan eksekusi fold sebelumnya.
    random.seed(SEED + fold); np.random.seed(SEED + fold)
    torch.manual_seed(SEED + fold); torch.cuda.manual_seed_all(SEED + fold)

    model = build_model(num_classes)
    model = apply_freeze_strategy(model)
    model = model.to(device)

    # Class weights dinormalisasi supaya rata-rata = 1 -- identik dengan baseline.
    class_counts  = np.bincount(train_labels, minlength=num_classes)
    class_weights = 1. / torch.tensor(class_counts, dtype=torch.float)
    class_weights = class_weights / class_weights.sum() * num_classes

    criterion = nn.CrossEntropyLoss(
        weight=class_weights.to(device), label_smoothing=LABEL_SMOOTHING
    )
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LR, weight_decay=WEIGHT_DECAY
    )
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=SCHEDULER_FACTOR, patience=SCHEDULER_PATIENCE,
        threshold=SCHEDULER_THRESHOLD, min_lr=SCHEDULER_MIN_LR
    )

    scaler = GradScaler()

    early_stopping = EarlyStopping(patience=EARLY_STOP_PATIENCE)
    best_val_f1 = -np.inf
    best_val_acc = -np.inf
    best_val_precision = -np.inf
    best_val_recall = -np.inf
    best_val_loss = np.inf
    best_train_loss = np.inf
    best_model_path = None
    train_losses, val_losses = [], []

    for epoch in range(EPOCHS):
        print(f"\n[{ARCH_KEY} | fold {fold+1}] Epoch {epoch+1}/{EPOCHS} (LR: {optimizer.param_groups[0]['lr']:.2e})")

        # TRAIN
        model.train()
        train_loss = 0
        for imgs, tgts in tqdm(train_loader, desc="Train"):
            imgs, tgts = imgs.to(device), tgts.to(device)
            optimizer.zero_grad()
            with autocast():
                loss = criterion(model(imgs), tgts)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item()

        # VALIDATION
        model.eval()
        val_loss = 0
        preds, trues = [], []
        with torch.no_grad():
            for imgs, tgts in tqdm(val_loader, desc="Val"):
                imgs, tgts = imgs.to(device), tgts.to(device)
                with autocast():
                    outputs = model(imgs)
                    v_loss  = criterion(outputs, tgts)
                val_loss += v_loss.item()
                preds.extend(outputs.argmax(1).cpu().numpy())
                trues.extend(tgts.cpu().numpy())

        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss   = val_loss / len(val_loader)

        acc       = accuracy_score(trues, preds)
        precision = precision_score(trues, preds, average="weighted", zero_division=0)
        recall    = recall_score(trues, preds, average="weighted", zero_division=0)
        f1        = f1_score(trues, preds, average="weighted", zero_division=0)

        scheduler.step(f1)

        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)

        print(f"Train Loss : {avg_train_loss:.4f} | Val Loss  : {avg_val_loss:.4f}")
        print(f"Accuracy   : {acc:.4f}  | Precision : {precision:.4f}")
        print(f"Recall     : {recall:.4f}  | F1 Score  : {f1:.4f}")

        run.log({
            "epoch": epoch + 1,
            f"fold_{fold+1}/train_loss": avg_train_loss,
            f"fold_{fold+1}/val_loss": avg_val_loss,
            f"fold_{fold+1}/accuracy": acc,
            f"fold_{fold+1}/precision": precision,
            f"fold_{fold+1}/recall": recall,
            f"fold_{fold+1}/f1_score": f1,
            f"fold_{fold+1}/lr": optimizer.param_groups[0]["lr"],
            # BARU (EXP03): pantau gamma SA module tiap epoch -- bagus untuk cek
            # apakah attention benar-benar "dipakai" (gamma menjauh dari 0) atau
            # tidak berkontribusi (gamma tetap ~0 sepanjang training).
            f"fold_{fold+1}/sa_gamma": model.attention.gamma.item(),
        })

        # SAVE BEST MODEL -- kriteria val_f1 tertinggi (bukan val_loss terendah)
        if f1 > best_val_f1:
            best_val_f1 = f1
            best_val_acc = acc
            best_val_precision = precision
            best_val_recall = recall
            best_val_loss = avg_val_loss
            best_train_loss = avg_train_loss

            save_path = f"{OUTPUT_DIR}/{ARCH_KEY}_fold{fold+1}.pth"
            torch.save({
                "model_state_dict": model.state_dict(),
                "val_loss": avg_val_loss,
                "f1": f1,
                "fold": fold + 1,
                "arch": ARCH_KEY,
            }, save_path)
            best_model_path = save_path
            print(f"  ✓ Model saved → {save_path} (F1: {f1:.4f})")

        if early_stopping.step(f1):
            print("Early Stopping Triggered")
            break

    # ── PLOT LOSS CURVE PER FOLD ──────────────────────────────────────────────
    epochs_ran = range(1, len(train_losses) + 1)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(epochs_ran, train_losses, label="Train Loss", marker="o", markersize=3)
    ax.plot(epochs_ran, val_losses, label="Val Loss", marker="o", markersize=3)
    ax.set_title(f"{ARCH_KEY} — Fold {fold+1} Loss Curve")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    ax.legend(); ax.grid(True, alpha=0.3)
    curve_path = f"{OUTPUT_DIR}/{ARCH_KEY}_Fold_{fold+1}_Loss_Curve.png"
    fig.savefig(curve_path, dpi=150, bbox_inches="tight")
    run.log({f"Loss_Curve/Fold_{fold+1}": wandb.Image(curve_path)})
    plt.close(fig)

    return {
        "arch": ARCH_KEY,
        "fold": fold + 1,
        "train_loss": best_train_loss,
        "val_loss": best_val_loss,
        "accuracy": best_val_acc,
        "precision": best_val_precision,
        "recall": best_val_recall,
        "f1": best_val_f1,
        "model_path": best_model_path,
    }


## 8. MAIN LOOP — 5 Fold (ResNet50 + SA)

Kalau waktu habis di tengah jalan: checkpoint tiap fold udah ke-save duluan (di `all_results`), aman buat dilanjut manual per-fold.

In [8]:
all_results = []

run = wandb.init(
    project="SkinDisease-CNN",
    entity="devianestnarendra_Team",
    name=f"{ARCH_KEY}",
    reinit=True,
    config={
        "architecture": ARCH_KEY,
        "n_folds": N_FOLDS,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "img_size": IMG_SIZE,
        "optimizer": "AdamW",
        "scheduler": f"ReduceLROnPlateau(mode=max, factor={SCHEDULER_FACTOR}, patience={SCHEDULER_PATIENCE})",
        "lr": LR,
        "dropout": DROPOUT,
        "weight_decay": WEIGHT_DECAY,
        "label_smoothing": LABEL_SMOOTHING,
        "unfreeze_patterns": UNFREEZE_PATTERNS,
        "checkpoint_criteria": "best_val_f1",
        "amp": True,
        "seed": SEED,
        # BARU (EXP03): catat konfigurasi SA module di W&B, supaya bisa ditelusuri
        # kalau nanti coba sweep reduction ratio yang lain.
        "sa_module": "SelfAttention2d (SAGAN-style, non-local)",
        "sa_reduction": SA_REDUCTION,
    }
)

for fold, (train_idx, val_idx) in enumerate(skf.split(filepaths, labels)):
    result = train_one_fold(fold, train_idx, val_idx, run)
    all_results.append(result)

run.finish()

results_df = pd.DataFrame(all_results)
results_df


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


C:\Users\UNIDA\AppData\Local\Temp\ipykernel_25680\3302125778.py:44: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


  [EXP03_ResNet50_SelfAttention] Trainable params: 28,575,256 / 28,800,600 (99.2%)

[EXP03_ResNet50_SelfAttention | fold 1] Epoch 1/50 (LR: 1.00e-04)


Train:   0%|          | 0/389 [00:00<?, ?it/s]C:\Users\UNIDA\AppData\Local\Temp\ipykernel_25680\3302125778.py:65: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Val:   0%|          | 0/98 [00:00<?, ?it/s]C:\Users\UNIDA\AppData\Local\Temp\ipykernel_25680\3302125778.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Val: 100%|██████████| 98/98 [00:40<00:00,  2.41it/s]


Train Loss : 3.1532 | Val Loss  : 3.0681
Accuracy   : 0.2265  | Precision : 0.2675
Recall     : 0.2265  | F1 Score  : 0.2028
  ✓ Model saved → outputs/EXP03_ResNet50_SelfAttention_fold1.pth (F1: 0.2028)

[EXP03_ResNet50_SelfAttention | fold 1] Epoch 2/50 (LR: 1.00e-04)


...

Train Loss : 1.0259 | Val Loss  : 2.3549
Accuracy   : 0.5583  | Precision : 0.5781
Recall     : 0.5583  | F1 Score  : 0.5619

[EXP03_ResNet50_SelfAttention | fold 5] Epoch 50/50 (LR: 1.00e-05)


Train:   0%|          | 0/389 [00:00<?, ?it/s]C:\Users\UNIDA\AppData\Local\Temp\ipykernel_25680\3302125778.py:65: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Val:   0%|          | 0/98 [00:00<?, ?it/s]C:\Users\UNIDA\AppData\Local\Temp\ipykernel_25680\3302125778.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Val: 100%|██████████| 98/98 [00:26<00:00,  3.70it/s]


Train Loss : 1.0207 | Val Loss  : 2.3520
Accuracy   : 0.5641  | Precision : 0.5837
Recall     : 0.5641  | F1 Score  : 0.5683
  ✓ Model saved → outputs/EXP03_ResNet50_SelfAttention_fold5.pth (F1: 0.5683)


epoch,▄▄▅▅▆▇▇▇█▁▂▃▃▃▅▇███▂▃▁▁▂▃▃▄▄▅▆██▁▂▃▄▄▆▆█
fold_1/accuracy,▁▂▂▃▃▄▄▅▄▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇█▇██████
fold_1/f1_score,▁▂▂▄▄▅▅▅▅▅▆▆▆▆▆▇▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇█████████
fold_1/lr,████████████████████████████████████▁▁▁▁
fold_1/precision,▁▂▄▄▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇█▇▇███████████
fold_1/recall,▁▂▂▃▃▄▄▄▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇█▇██████
fold_1/sa_gamma,▁▂▂▂▃▃▄▄▄▅▅▅▆▆▆▇▇▇▇▇▇███████████████████
fold_1/train_loss,█▇▆▆▆▅▅▅▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
fold_1/val_loss,█▇▆▅▅▄▄▄▄▄▄▃▃▃▃▃▃▂▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▂▂▁▁▁
fold_2/accuracy,▁▂▂▃▃▃▄▄▄▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇███▇████████████
+31,...


,arch,fold,train_loss,val_loss,accuracy,precision,recall,f1,model_path
0,EXP03_ResNet50_SelfAttention,1,0.995145,2.304383,0.580013,0.589887,0.580013,0.581715,outputs/EXP03_ResNet50_SelfAttention_fold1.pth
1,EXP03_ResNet50_SelfAttention,2,1.023290,2.352796,0.560411,0.578635,0.560411,0.563551,outputs/EXP03_ResNet50_SelfAttention_fold2.pth
2,EXP03_ResNet50_SelfAttention,3,1.200175,2.449208,0.530698,0.551026,0.530698,0.533085,outputs/EXP03_ResNet50_SelfAttention_fold3.pth
3,EXP03_ResNet50_SelfAttention,4,0.990716,2.282766,0.570878,0.580138,0.570878,0.572479,outputs/EXP03_ResNet50_SelfAttention_fold4.pth
4,EXP03_ResNet50_SelfAttention,5,1.020732,2.351968,0.564127,0.583726,0.564127,0.568256,outputs/EXP03_ResNet50_SelfAttention_fold5.pth


## 9. Rekap 5-Fold

In [9]:
print("\n" + "="*50)
print(f"  FINAL RESULT — ALL FOLDS ({ARCH_KEY})")
print("="*50)
print(f"Mean Accuracy  : {results_df['accuracy'].mean():.4f} ± {results_df['accuracy'].std():.4f}")
print(f"Mean Precision : {results_df['precision'].mean():.4f} ± {results_df['precision'].std():.4f}")
print(f"Mean Recall    : {results_df['recall'].mean():.4f} ± {results_df['recall'].std():.4f}")
print(f"Mean F1 Score  : {results_df['f1'].mean():.4f} ± {results_df['f1'].std():.4f}")

# ── WANDB LOG SUMMARY ───────────────────────────────────────────────────────
try:
    run.log({
        "summary/mean_accuracy"  : results_df["accuracy"].mean(),
        "summary/mean_precision" : results_df["precision"].mean(),
        "summary/mean_recall"    : results_df["recall"].mean(),
        "summary/mean_f1"        : results_df["f1"].mean(),
        "summary/std_accuracy"   : results_df["accuracy"].std(),
        "summary/std_precision"  : results_df["precision"].std(),
        "summary/std_recall"     : results_df["recall"].std(),
        "summary/std_f1"         : results_df["f1"].std(),
    })
except Exception as e:
    print(f"  ⚠ W&B log summary gagal (dilewati): {e}")

# ── GRAFIK: 4 metrik per fold ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
metric_cols = ["accuracy", "precision", "recall", "f1"]
metric_titles = ["Accuracy", "Precision", "Recall", "F1-Score"]

for ax, col, title in zip(axes, metric_cols, metric_titles):
    ax.bar(results_df["fold"].astype(str), results_df[col], color="#1f3a5f")
    ax.axhline(results_df[col].mean(), color="red", linestyle="--", label=f"Mean = {results_df[col].mean():.3f}")
    ax.set_xlabel("Fold"); ax.set_ylabel(f"Val {title}")
    ax.set_title(f"{ARCH_KEY} — {title} per Fold")
    ax.set_ylim(0, 1); ax.legend()

plt.tight_layout()



  FINAL RESULT — ALL FOLDS (EXP03_ResNet50_SelfAttention)
Mean Accuracy  : 0.5612 ± 0.0186
Mean Precision : 0.5767 ± 0.0150
Mean Recall    : 0.5612 ± 0.0186
Mean F1 Score  : 0.5638 ± 0.0184
  ⚠ W&B log summary gagal (dilewati): Run (eq1dww9i) is finished. The call to `log` will be ignored. Please make sure that you are using an active run.


## 10. Test Evaluation (Fold Terbaik)

In [10]:
best_fold_result = max(all_results, key=lambda r: r["f1"])
best_overall_path = best_fold_result["model_path"]
print(f"Fold terbaik    : {best_fold_result['fold']}")
print(f"Checkpoint      : {best_overall_path}")
print(f"Val F1 terbaik  : {best_fold_result['f1']:.4f}")

_, eval_tf = get_transforms(IMG_SIZE)
test_dataset = datasets.ImageFolder(TEST_DIR, transform=eval_tf)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

model = build_model(num_classes)
model = model.to(device)
checkpoint = torch.load(best_overall_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

y_true, y_pred = [], []
with torch.no_grad():
    for imgs, tgts in tqdm(test_loader, desc="Test"):
        imgs = imgs.to(device)
        with autocast():
            out = model(imgs)
        y_true.extend(tgts.numpy())
        y_pred.extend(out.argmax(1).cpu().numpy())

acc = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average="weighted", zero_division=0)
recall = recall_score(y_true, y_pred, average="weighted", zero_division=0)
f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

print(f"Accuracy  : {acc:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-Score  : {f1:.4f}")
print()
print(classification_report(y_true, y_pred, target_names=classes, zero_division=0))

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(15, 15))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes).plot(cmap="Blues", ax=ax, xticks_rotation=90)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/{ARCH_KEY}_Test_Confusion_Matrix.png", dpi=150, bbox_inches="tight")
plt.show()

test_summary_df = pd.DataFrame([{
    "arch": ARCH_KEY, "Accuracy": acc, "Precision": precision, "Recall": recall, "F1": f1
}])
test_summary_df.to_csv(f"{OUTPUT_DIR}/{ARCH_KEY}_Test_Summary.csv", index=False)
print(f"\n✓ Test summary disimpan -> {OUTPUT_DIR}/{ARCH_KEY}_Test_Summary.csv")


Fold terbaik    : 1
Checkpoint      : outputs/EXP03_ResNet50_SelfAttention_fold1.pth
Val F1 terbaik  : 0.5817


Test:   0%|          | 0/126 [00:00<?, ?it/s]C:\Users\UNIDA\AppData\Local\Temp\ipykernel_25680\3000596476.py:21: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Test: 100%|██████████| 126/126 [00:48<00:00,  2.61it/s]


Accuracy  : 0.5820
Precision : 0.5956
Recall    : 0.5820
F1-Score  : 0.5837

                                                                    precision    recall  f1-score   support

                                           Acne and Rosacea Photos       0.78      0.86      0.82       312
Actinic Keratosis Basal Cell Carcinoma and other Malignant Lesions       0.72      0.54      0.62       288
                                          Atopic Dermatitis Photos       0.47      0.61      0.53       123
                                            Bullous Disease Photos       0.57      0.49      0.53       113
                Cellulitis Impetigo and other Bacterial Infections       0.33      0.48      0.39        73
                                                     Eczema Photos       0.60      0.53      0.56       309
                                      Exanthems and Drug Eruptions       0.42      0.52      0.47       101
                 Hair Loss Photos Alopecia and other Hair 

C:\Users\UNIDA\AppData\Local\Temp\ipykernel_25680\3000596476.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 11. Visualisasi Attention Map (Bonus, khusus EXP03)

Sel opsional -- bukan bagian dari metrik utama, tapi berguna untuk interpretability:
menampilkan overlay attention map SA module di atas beberapa gambar test, supaya bisa dicek
apakah attention memang fokus ke area lesi/kulit yang relevan, atau malah ke background/artefak.

In [11]:
import torch.nn.functional as F

def visualize_sa_attention(model, dataset, class_names, n_samples=6, save_path=None):
    model.eval()
    fig, axes = plt.subplots(2, n_samples, figsize=(3 * n_samples, 6))

    idxs = random.sample(range(len(dataset)), n_samples)
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    with torch.no_grad():
        for col, idx in enumerate(idxs):
            img_tensor, label = dataset[idx]
            inp = img_tensor.unsqueeze(0).to(device)

            with autocast():
                logits, attn = model(inp, return_attention=True)
            pred = logits.argmax(1).item()

            # attn: [1, N, N] -- rata-ratakan "seberapa besar tiap posisi diperhatikan
            # oleh posisi lain" (mean over query dim) -> peta relevansi [H, W]
            H = W = int(attn.shape[-1] ** 0.5)
            attn_map = attn[0].mean(dim=0).view(H, W).cpu()
            attn_map = F.interpolate(
                attn_map.unsqueeze(0).unsqueeze(0), size=(IMG_SIZE, IMG_SIZE),
                mode="bilinear", align_corners=False
            ).squeeze().numpy()

            img_denorm = (img_tensor * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

            axes[0, col].imshow(img_denorm)
            axes[0, col].set_title(f"GT: {class_names[label]}", fontsize=8)
            axes[0, col].axis("off")

            axes[1, col].imshow(img_denorm)
            axes[1, col].imshow(attn_map, cmap="jet", alpha=0.5)
            axes[1, col].set_title(f"Pred: {class_names[pred]}", fontsize=8)
            axes[1, col].axis("off")

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


visualize_sa_attention(
    model, test_dataset, classes, n_samples=6,
    save_path=f"{OUTPUT_DIR}/{ARCH_KEY}_Attention_Visualization.png"
)
print(f"✓ Visualisasi attention disimpan -> {OUTPUT_DIR}/{ARCH_KEY}_Attention_Visualization.png")


C:\Users\UNIDA\AppData\Local\Temp\ipykernel_25680\3458500898.py:16: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


✓ Visualisasi attention disimpan -> outputs/EXP03_ResNet50_SelfAttention_Attention_Visualization.png


C:\Users\UNIDA\AppData\Local\Temp\ipykernel_25680\3458500898.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Catatan

- Checkpoint format: `{model_state_dict, val_loss, f1, fold, arch}` -- tetap kompatibel dengan
  baseline & notebook arsitektur lain untuk rencana perbandingan multi-arsitektur.
- `results_df` punya kolom yang sama (`arch, fold, train_loss, val_loss, accuracy, precision,
  recall, f1, model_path`) -- tinggal `pd.concat()` dengan hasil EXP01 baseline saat rekap akhir.
- **Yang berbeda dari baseline**: hanya arsitektur head (`SelfAttention2d` sebelum global pool +
  FC). Training regime, augmentasi, scheduler, seed control, dan kriteria checkpoint identik --
  supaya selisih F1/Accuracy antara EXP01 vs EXP03 bisa diatribusikan ke modul attention, bukan
  confound lain.
- `sa_gamma` di-log tiap epoch per fold -- kalau nilainya tetap mendekati 0 sampai akhir training,
  itu indikasi attention module belum banyak berkontribusi (bisa jadi perlu LR terpisah untuk
  attention, atau lebih banyak epoch sebelum early-stop).
- **Rekomendasi lanjutan**: kalau ingin membandingkan varian attention lain (mis. CBAM, channel
  attention SE-block, atau attention di beberapa stage bukan cuma setelah `layer4`), cukup ganti
  isi `SelfAttention2d`/tempat modul disisipkan di `ResNetSA` -- kerangka training (`train_one_fold`,
  main loop, rekap, test eval) tidak perlu diubah.
